## AgentCore Evaluations - on-demand evaluation for Strands Agent

In this tutorial you will learn about to use the on-demand evaluation from AgentCore Evaluations applied to a Strands agent.

To execute this lab you should first have created the Strands agent using the code at [00-prereqs](../../00-prereqs) folder and created your custom evaluator using the code at [01-creating-custom-evaluators](../../01-creating-custom-evaluators)

### What You'll Learn
- How to run on-demand evaluations to a trace using the AgentCore SDK and boto3

### Tutorial Details

| Information         | Details                                                                       |
|:--------------------|:------------------------------------------------------------------------------|
| Tutorial type       | Evaluating Strands agent with on-demand evaluators (built-in and custom)      |
| Tutorial components | Running evaluation with built-in and custom evaluators                        |
| Tutorial vertical   | Cross-vertical                                                                |
| Example complexity  | Easy                                                                          |
| SDK used            | Amazon Bedrock AgentCore SDK / boto3                                          |

### On-demand evaluation

On-demand evaluation provides a flexible way to evaluate specific agent interactions by directly analyzing a chosen set of spans. Unlike online evaluation which continuously monitors production traffic, on-demand evaluation lets you perform targeted assessments of selected interactions at any time.

With on-demand evaluation, you specify the exact spans, traces or sessions you want to evaluate by providing their span, trace or session IDs. You can automatically evaluate all traces in a session by providing the session ID to `EvaluationClient.run()`.

You can then apply custom evaluators or built-in evaluators to your agent's interactions. This evaluation type is particularly useful when you need to investigate specific customer interactions, validate fixes for reported issues, or analyze historical data for quality improvements. Once you submit the evaluation request, the service processes only the specified spans and provides detailed results for your analysis.

### Generating traces on AgentCore Observability from an agent

AgentCore Observability provides comprehensive visibility into agent behavior during invocations by leveraging [OpenTelemetry (OTEL)](https://opentelemetry.io/) traces as the foundation for capturing and structuring detailed execution data. AgentCore relies on [AWS Distro for OpenTelemetry (ADOT)](https://aws-otel.github.io/) to instrument different types of OTEL traces across various agent frameworks.

When your agent is hosted on AgentCore Runtime (like our agent in this tutorial), the AgentCore Observability instrumentation is automatic, with minimal configuration. All you need to do is include `aws-opentelemetry-distro` in `requirements.txt` and AgentCore Runtime handles OTEL configuration automatically. When your agent is not running in AgentCore Runtime, you will need to instrument it with ADOT to have it available in AgentCore Observability. You need to configure environment variables to direct telemetry data to CloudWatch and run your agent with OpenTelemetry instrumentation.

The process looks as following:

![session_traces](../../images/observability_traces.png)

Once your session traces are available in AgentCore Observability, you can use AgentCore Evaluations to evaluate your agent's behavior.

### How on-demand evaluation works with the traces

On the on-demand evaluation, your agent is invoked and generates traces in AgentCore Observability. Those traces are mapped to sessions and their logs are made available in Amazon CloudWatch Log groups. With the on-demand evaluation, a developer decides which sessions or traces to use and sends those as inputs to AgentCore Evaluations, together with the metrics to evaluate the traces content. The process looks as following:


![session_traces](../../images/on_demand_evaluations.png)

### Retrieving information from previous tutorials

For this tutorial, we will use the Strands agent deployed in AgentCore Runtime during our prerequisites tutorial. We will evaluate it with pre-built metrics and with the `response_quality` metric we created in the `01-creating-custom-metrics` tutorial. Let's retrieve our agent and evaluator informations.

In [1]:
%store -r agent_id_strands
%store -r agent_arn_strands
%store -r session_id_strands
%store -r evaluator_id
try:
    print("Agent Id:", agent_id_strands)
    print("Agent ARN:", agent_arn_strands)
except NameError:
    raise Exception(
        """Missing agent info from your Strands agent. Please run 00-prereqs before executing this lab"""
    )

try:
    print("Session id:", session_id_strands)
except NameError:
    raise Exception(
        """Missing session id from your Strands agent. Please run 00-prereqs before executing this lab"""
    )

try:
    print("Evaluator id:", evaluator_id)
except NameError:
    raise Exception(
        """Missing custom evaluator id. Please run 01-creating-custom-evaluators before executing this lab"""
    )

Agent Id: acevalstrands2-xKJy20HJDc
Agent ARN: arn:aws:bedrock-agentcore:us-east-1:849138760372:runtime/acevalstrands2-xKJy20HJDc
Session id: cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb
Evaluator id: response_quality_for_scope_b40a1c32-vBGSJi4Nz4


### Initiating the AgentCore Evaluations client

Now let's initiate the AgentCore Evaluations client using the `EvaluationClient` from the `bedrock_agentcore` SDK. This client provides a high-level interface for running on-demand evaluations.

**On-demand evaluations** let you score a specific agent session after the fact. You supply a session ID, the runtime ARN, and one or more evaluators. AgentCore reads the session's OpenTelemetry spans from CloudWatch Logs, scores each turn against the evaluator rubric, and returns per-turn and aggregate scores — no changes to your agent code required.

> **Learn more:** [Run on-demand evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/on-demand-evaluation.html)


In [2]:
from bedrock_agentcore.evaluation import EvaluationClient
from datetime import timedelta
import os
import json
from boto3.session import Session
from IPython.display import Markdown, display

In [3]:
boto_session = Session()
region = boto_session.region_name
print(region)

us-east-1


In [4]:
eval_client = EvaluationClient(region_name=region)

### Running evaluations

To run AgentCore Evaluations, you must provide session, trace or span information. Different metrics require different levels of information from your agent traces, as we saw in the previous tutorial.

![metrics level](../../images/metrics_per_level.png)

`EvaluationClient.run()` takes a session ID and automatically extracts and evaluates all the traces in that session for trace and span level metrics. You provide the `agent_id`, `session_id`, and a `look_back_time` window for CloudWatch log retrieval. Results are returned as a list of dicts with keys such as `label`, `value`, `explanation`, and `evaluatorId`.

### Goal Success Rate

Let's now evaluate the Goal Success Rate of our agent. Remember, we asked the agent the following questions:

* What is the weather now?
* How much is 2+2?
* Can you tell me the capital of the US?

In [5]:
goal_success_results = eval_client.run(
    evaluator_ids=["Builtin.GoalSuccessRate"],
    agent_id=agent_id_strands,
    session_id=session_id_strands,
    look_back_time=timedelta(hours=24),
)

Let's now understand the results of our evaluator. The results object contains the information about the evaluation performed (session_id, trace_id, input_data) as well as the results from the evaluation.

The evaluation results include the evaluator information (id, name, ARN), the evaluation value, the evaluation label, the evaluation explanation and some extra context about the evaluation job (spanContext, token_usage, ...).

Let's take a look at our evaluation response

In [6]:
for result in goal_success_results:
    information = f"""
    Goal Success: {result.get("label", result.get("rating", "N/A"))} ({result.get("value", result.get("score", "N/A"))})
    Explanation: \n{result.get("explanation", result.get("reason", ""))}\n
    Token Usage: {result.get("token_usage", {})}\n
    Context: {result.get("context", {})}\n
    """
    display(Markdown(information))


    Goal Success: Yes (1.0)
    Explanation: 
The conversation contains three user goals:

1. **Calculate 2+2**: The user asked for a calculation. The assistant correctly used the calculator tool with expression '2+2', received the result '4', and responded with '2 + 2 = **4**'. This goal was successfully achieved.

2. **Check current weather**: The user asked about the current weather. The assistant correctly used the weather tool, received 'sunny' as the result, and responded with 'The weather right now is **sunny**! ☀️'. This goal was successfully achieved.

3. **Ask for the capital of the US**: The user asked for the capital of the United States. The assistant correctly responded with 'Washington, D.C.' without using any tools (as no geography/knowledge tool was available). The assistant provided accurate information and even acknowledged the limitation of available tools while still answering the question correctly. This goal was successfully achieved.

All three user queries were addressed appropriately. The assistant used tools when available and appropriate (calculator and weather), and provided accurate information from its knowledge base when no relevant tool was available (US capital question). Each response was accurate and directly answered the user's question.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb'}}

    

### Correctness

Let's now analyze the same session for a trace-level metric: Correctness

In [7]:
correctness_results = eval_client.run(
    evaluator_ids=["Builtin.Correctness"],
    agent_id=agent_id_strands,
    session_id=session_id_strands,
    look_back_time=timedelta(hours=24),
)

Let's now understand the results of our evaluator. In this case, correctness is evaluated at a trace level, so each trace will get its own evaluation result. 

In [8]:
for result in correctness_results:
    information = f"""
    Correctness: {result.get("label", result.get("rating", "N/A"))} ({result.get("value", result.get("score", "N/A"))})
    Explanation: \n{result.get("explanation", result.get("reason", ""))}\n
    Token Usage: {result.get("token_usage", {})}\n
    Context: {result.get("context", {})}\n
    """
    display(Markdown(information))
    print("================================================")


    Correctness: Perfectly Correct (1.0)
    Explanation: 
The task asks 'How much is 2+2?' The assistant used a calculator tool which returned 'Result: 4'. The candidate response states '2 + 2 = **4**'. This directly answers the question with the correct value of 4, which matches the tool output. The mathematical calculation is accurate, and the response provides the exact answer requested. The formatting with bold text does not affect the correctness of the mathematical content.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb', 'traceId': '69fa5bc577757bcd08f9abb254606659'}}

    


    Correctness: Perfectly Correct (1.0)
    Explanation: 
The task is to evaluate whether the Assistant's response correctly answers the question 'How is the weather now?'. Looking at the context, a weather tool was called and returned the result 'sunny'. The Assistant's response states 'The weather right now is **sunny**! ☀️'. This directly matches the tool output, which according to the instructions takes priority over any other knowledge. The Assistant correctly interpreted the tool result and provided an accurate answer to the user's question. The addition of formatting (bold text) and an emoji are stylistic choices that don't affect the correctness of the factual content. Since the response accurately conveys the information from the tool output ('sunny'), it is a correct answer to the question asked.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb', 'traceId': '69fa5bcf1dd602f241c7c9257ee63be6'}}

    


    Correctness: Perfectly Correct (1.0)
    Explanation: 
The task asks for the capital of the United States. The Assistant's response states that the capital is Washington, D.C. (Washington, District of Columbia), which is factually correct and accurate. This is a straightforward general knowledge question that has been answered correctly. The additional commentary about not having specialized tools for general knowledge questions is supplementary information that doesn't affect the correctness of the answer itself. Since the question was answered accurately and completely, this response is perfectly correct.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb', 'traceId': '69fa5bd30ad6fbc16c9c08604c35f5f2'}}

    

### Tool selection accuracy and parameter selection accuracy

Let's now evaluate our agent for the tool and parameter selection. Both metrics are evaluated at the span level.

In [9]:
parameter_results = eval_client.run(
    evaluator_ids=["Builtin.ToolParameterAccuracy", "Builtin.ToolSelectionAccuracy"],
    agent_id=agent_id_strands,
    session_id=session_id_strands,
    look_back_time=timedelta(hours=24),
)

Let's now analyze the results. In this case, we are evaluating the session with two different metrics in the same run. That means that we now need to know which evaluator is producing each response. We can do that with the `evaluator_name` property of the result. Let's see how well our agent used tools:

In [10]:
for result in parameter_results:
    information = f"""
    Metric: {result.get("evaluatorId", "")}
    Value: {result.get("label", result.get("rating", "N/A"))} ({result.get("value", result.get("score", "N/A"))})
    Explanation: \n{result.get("explanation", result.get("reason", ""))}\n
    Token Usage: {result.get("token_usage", {})}\n
    Context: {result.get("context", {})}\n
    """
    display(Markdown(information))
    print("================================================")


    Metric: Builtin.ToolParameterAccuracy
    Value: Yes (1.0)
    Explanation: 
The tool-call uses the 'calculator' tool with a single parameter 'expression' set to '2+2'. 

Parameter Analysis:
1. 'expression': '2+2' - This value comes directly from the user's question "How much is 2+2?". The user explicitly asked about the calculation "2+2", and this exact expression has been faithfully transferred to the parameter value. The parameter format appears correct as a string representation of a mathematical expression.

The parameter value is not fabricated or hallucinated - it is a direct extraction from the user's input. The user asked to calculate "2+2" and the agent passed exactly "2+2" as the expression parameter to the calculator tool. This is a faithful representation of what the user requested.

No optional parameters appear to be omitted based on the available information. The tool-call contains the necessary information to fulfill the user's request using only data from the preceding context.

The parameter follows a reasonable format for a calculator expression (a string containing numbers and operators), and the value has clear traceability to the user's original question.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb', 'traceId': '69fa5bc577757bcd08f9abb254606659', 'spanId': '358504e3839dac22'}}

    


    Metric: Builtin.ToolParameterAccuracy
    Value: Yes (1.0)
    Explanation: 
The user asked 'How is the weather now?' which is a straightforward request for current weather information. The tool-call invokes the 'weather' function with an empty parameters object: `'parameters': {}`.

Analyzing the parameters:
1. The weather tool is called with no parameters provided
2. The user's question does not specify any location, time, or other details - just asks about 'the weather now'
3. Since no parameters are explicitly provided in the tool-call, there are no parameter values to trace back to the context
4. The empty parameters object `{}` indicates no specific values were fabricated or hallucinated

The key question is whether omitting all parameters (assuming the weather API has optional parameters like location) is faithful to the context. The user did not provide:
- A specific location
- A specific time (though 'now' implies current time)
- Any other weather-related specifications

Since the tool-call passes an empty parameters object and doesn't fabricate any values (like inventing a location the user never mentioned), this is faithful to the context. The assistant is not hallucinating parameter values - it's simply calling the weather tool without specifying parameters that weren't provided by the user. Whether this will successfully return weather information is not our concern; we only evaluate if parameters are faithful to context.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb', 'traceId': '69fa5bcf1dd602f241c7c9257ee63be6', 'spanId': 'a078a5d0247203b7'}}

    


    Metric: Builtin.ToolSelectionAccuracy
    Value: Yes (1.0)
    Explanation: 
The user explicitly asked 'How much is 2+2?' which is a straightforward arithmetic question. The assistant has access to a calculator tool and used it to compute the expression '2+2', receiving the correct result of 4. This action is completely justified because: (1) The user's request is clearly a mathematical calculation that requires computing a sum, (2) The calculator tool is specifically designed for such operations, (3) The parameters provided ('2+2') directly match the user's question, (4) The tool successfully returned the correct answer (4), which directly addresses the user's need. There is no ambiguity in the user's intent - they want to know the result of adding 2 and 2. Using the calculator tool is the most appropriate and efficient way to provide an accurate answer. A helpful assistant would reasonably take this action to serve the user's explicit request for a mathematical calculation.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb', 'traceId': '69fa5bc577757bcd08f9abb254606659', 'spanId': '358504e3839dac22'}}

    


    Metric: Builtin.ToolSelectionAccuracy
    Value: Yes (1.0)
    Explanation: 
The user explicitly asks 'How is the weather now?' which is a direct request for current weather information. The assistant has access to a weather tool that can provide this information. The tool call to 'weather' directly addresses the user's question and is the appropriate action to take in this context.

While the weather tool appears to have been called with empty parameters ({}), this may be acceptable depending on the tool's design - it could default to the user's current location or use contextual information. The tool successfully returned a result ('sunny'), indicating that the call was functional despite the minimal parameters.

The action is:
1. Directly responsive to the user's explicit request
2. Aligned with the user's clear intent to know the current weather
3. Functional, as evidenced by the successful return of weather data
4. Exactly what a helpful assistant should do when asked about weather

This is a straightforward case where the user asks a question, and the assistant uses the appropriate tool to answer it. The action is fully justified.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb', 'traceId': '69fa5bcf1dd602f241c7c9257ee63be6', 'spanId': 'a078a5d0247203b7'}}

    

### Using custom evaluator

Now that we have used evaluators in the session, trace and span level, let's use our custom metric to evaluate our response quality:

In [11]:
custom_results = eval_client.run(
    evaluator_ids=[evaluator_id],
    agent_id=agent_id_strands,
    session_id=session_id_strands,
    look_back_time=timedelta(hours=24),
)

Let's now take a look at the evaluation results. In this case, we are evaluating an agent that has the following instructions:

```
You're a helpful assistant. You can do simple math calculation, and tell the weather.
```

For our evaluation metric we are penalizing the agent for going out of scope with a `Very Poor` quality as stated in our evaluation instructions:

```
...
**IMPORTANT**: A response quality can only be high if the agent remains in its original scope. Penalize agents that answer questions outside its original scope with a Very Poor classification.
...
```

Since we are evaluating the following questions:

* What is the weather now?
* How much is 2+2?
* Can you tell me the capital of the US?

We expect the agent to have a `Very Poor` evaluation for the last question.

In [12]:
for result in custom_results:
    information = f"""
    Metric: {result.get("evaluatorId", "")}
    Value: {result.get("label", result.get("rating", "N/A"))} ({result.get("value", result.get("score", "N/A"))})
    Explanation: \n{result.get("explanation", result.get("reason", ""))}\n
    Token Usage: {result.get("token_usage", {})}\n
    Context: {result.get("context", {})}\n
    """
    display(Markdown(information))
    print("================================================")


    Metric: response_quality_for_scope_b40a1c32-vBGSJi4Nz4
    Value: Very Good (1.0)
    Explanation: 
The task asks for the evaluation of a mathematical query: 'How much is 2+2?'. The candidate response states '2 + 2 = **4**', which is completely accurate. The mathematical calculation is correct, and the answer directly addresses the question asked. The context shows that a calculator tool was used to compute the result, which returned '4', and the assistant correctly reported this answer. Since this is a mathematical query and the assistant is within its allowed scope (weather and mathematical queries), there is no scope violation. The response is clear, concise, and provides the exact correct answer with no errors, omissions, or misleading information. The formatting with bold emphasis does not detract from the accuracy of the content.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb', 'traceId': '69fa5bc577757bcd08f9abb254606659'}}

    


    Metric: response_quality_for_scope_b40a1c32-vBGSJi4Nz4
    Value: Very Good (1.0)
    Explanation: 
The task is to evaluate whether the Assistant's response is accurate and appropriate given its scope of answering questions about weather and mathematical queries only.

Looking at the context:
1. The user asked 'How is the weather now?'
2. The weather tool was called and returned 'sunny'
3. The Assistant responded with 'The weather right now is **sunny**! ☀️'

Evaluation:
- The question is about weather, which is within the Assistant's defined scope
- The tool result shows the weather is 'sunny'
- The Assistant's response accurately conveys this information by stating 'The weather right now is **sunny**!'
- The response is factually correct based on the tool result
- The response directly answers the user's question
- The Assistant stayed within its designated scope (weather and math)
- The addition of the sun emoji is a presentational element that doesn't affect accuracy

The response is completely accurate, directly answers the question asked, uses the correct information from the tool result, and the Assistant properly remained within its scope of handling weather and math queries. There are no errors, omissions, or inaccuracies in the content of the response.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb', 'traceId': '69fa5bcf1dd602f241c7c9257ee63be6'}}

    


    Metric: response_quality_for_scope_b40a1c32-vBGSJi4Nz4
    Value: Very Poor (0.0)
    Explanation: 
The instructions explicitly state that the response quality can only be high if the agent remains in its original scope to answer questions about the weather and mathematical queries only. The instructions further specify to penalize agents that answer questions outside this scope with a Very Poor classification.

In this case, the user asked about the capital of the United States, which is neither a weather question nor a mathematical query. The assistant responded by providing the correct answer (Washington, D.C.), and even acknowledged that it doesn't have specialized tools for general knowledge questions and that it's primarily equipped with calculator and weather tools.

While the factual answer provided is accurate (Washington, D.C. is indeed the capital of the United States), the assistant violated its scope by answering a general knowledge question. According to the evaluation criteria, this breach of scope should result in a Very Poor classification, regardless of the accuracy of the information provided.

The assistant should have declined to answer the question or explicitly stated it can only help with weather and math-related queries.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': 'cedfa479-dccd-42b0-b8f8-5bd1d5dfecfb', 'traceId': '69fa5bd30ad6fbc16c9c08604c35f5f2'}}

    

### Saving evaluation results

You can save the evaluation results to a file for later analysis. Serialize the results list directly to JSON.

In [13]:
os.makedirs("evals_results", exist_ok=True)

save_results = eval_client.run(
    evaluator_ids=[evaluator_id],
    agent_id=agent_id_strands,
    session_id=session_id_strands,
    look_back_time=timedelta(hours=24),
)

with open("evals_results/output.json", "w") as f:
    json.dump(save_results, f, indent=2, default=str)
print("Results saved to evals_results/output.json")

Results saved to evals_results/output.json


### Congrats!

You have now evaluated your agent with the on-demand capabilities. In the next tutorial, we will automate the evaluation of the agent for a production environment by setting an online evaluator and connecting it with the agent.